# Exploracion de KNN para predecir `consumo_total`

Este notebook usa la tabla exportada `tabla_limpia_resultado.csv` y prueba valores de `k` desde `1` hasta `100` con `KNeighborsRegressor`.

La comparacion se hace con la misma logica base:
- variable objetivo: `consumo_total`
- predictores: `tot_hog`, `nivel_promedio_ponderado`, `diversidad_shannon`
- estandarizacion previa para KNN
- particion `train/test`


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid')


In [ ]:
# Cargamos la tabla limpia con componentes ya exportada desde el notebook principal.
ruta_csv = Path('consumo_agua/tabla_limpia_resultado.csv')

if not ruta_csv.exists():
    raise FileNotFoundError(
        'No existe consumo_agua/tabla_limpia_resultado.csv. '
        'Primero reejecuta en agua_cdmx_pca.ipynb la celda que construye '
        'tabla_limpia_resultado y lo exporta a CSV.'
    )

tabla_limpia_resultado = pd.read_csv(ruta_csv)
print(f'Tabla cargada desde: {ruta_csv.resolve()}')
print('Dimensiones:', tabla_limpia_resultado.shape)
tabla_limpia_resultado.head()


In [ ]:
# Preparamos la tabla para modelado.
variables_predictoras = [
    'tot_hog',
    'nivel_promedio_ponderado',
    'diversidad_shannon'
]
variable_objetivo = 'consumo_total'

tabla_modelo = tabla_limpia_resultado.dropna(
    subset=variables_predictoras + [variable_objetivo]
).copy()

X = tabla_modelo[variables_predictoras]
y = tabla_modelo[variable_objetivo]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler_knn = StandardScaler()
X_train_knn = scaler_knn.fit_transform(X_train)
X_test_knn = scaler_knn.transform(X_test)

print('Filas para modelado:', len(tabla_modelo))
print('Train:', X_train.shape, 'Test:', X_test.shape)


In [ ]:
# Probamos k desde 1 hasta 100 y guardamos metricas para cada modelo.
resultados_knn = []

for k in range(1, 101):
    modelo_knn = KNeighborsRegressor(
        n_neighbors=k,
        weights='distance',
        metric='minkowski',
        p=2
    )
    modelo_knn.fit(X_train_knn, y_train)
    y_pred_knn = modelo_knn.predict(X_test_knn)

    resultados_knn.append({
        'k': k,
        'R2': r2_score(y_test, y_pred_knn),
        'MAE': mean_absolute_error(y_test, y_pred_knn),
        'RMSE': mean_squared_error(y_test, y_pred_knn) ** 0.5
    })

resultados_knn = pd.DataFrame(resultados_knn)
resultados_knn.sort_values('R2', ascending=False).head(10)


In [ ]:
# Mejor valor de k segun R2.
mejor_resultado_knn = resultados_knn.sort_values('R2', ascending=False).iloc[0]
mejor_k = int(mejor_resultado_knn['k'])

print('Mejor k:', mejor_k)
print(mejor_resultado_knn)


In [ ]:
# Visualizamos como cambia el R2 conforme cambia k.
plt.figure(figsize=(12, 5))
sns.lineplot(data=resultados_knn, x='k', y='R2', marker='o', linewidth=1)
plt.axvline(mejor_k, color='tomato', linestyle='--', label=f'Mejor k = {mejor_k}')
plt.title('R2 de KNN para k de 1 a 100')
plt.xlabel('Numero de vecinos (k)')
plt.ylabel('R2 en test')
plt.legend()
plt.show()


In [ ]:
# Ajustamos de nuevo el mejor modelo y revisamos predicciones.
modelo_knn_final = KNeighborsRegressor(
    n_neighbors=mejor_k,
    weights='distance',
    metric='minkowski',
    p=2
)

modelo_knn_final.fit(X_train_knn, y_train)
y_pred_knn_final = modelo_knn_final.predict(X_test_knn)

comparacion_predicciones_knn = pd.DataFrame({
    'consumo_real': y_test.values,
    'consumo_predicho_knn': y_pred_knn_final,
    'error_knn': y_test.values - y_pred_knn_final
}).reset_index(drop=True)

comparacion_predicciones_knn.head(15)


In [ ]:
# Grafica de consumo real vs consumo predicho para el mejor k.
plt.figure(figsize=(7, 7))
sns.scatterplot(
    x=y_test,
    y=y_pred_knn_final,
    s=55,
    alpha=0.75
)

limite_min = min(y_test.min(), y_pred_knn_final.min())
limite_max = max(y_test.max(), y_pred_knn_final.max())
plt.plot([limite_min, limite_max], [limite_min, limite_max], linestyle='--', color='gray')
plt.title(f'Consumo real vs consumo predicho - KNN (k = {mejor_k})')
plt.xlabel('Consumo real')
plt.ylabel('Consumo predicho')
plt.show()
